# Discount Impact on Sales – Econometric Analysis

## Project Overview
This project investigates how discounts influence sales performance within a large-scale beverage retail dataset containing nearly 9 million transactions. Using Python and econometric modelling techniques, the analysis evaluates the relationship between discount levels, product characteristics, customer type, and sales outcomes.

The project applies an Ordinary Least Squares (OLS) regression model to quantify the sensitivity of sales to discounts while controlling for customer behaviour and product-level effects.

---

# Business Problem
Retailers frequently use discounts to increase demand, improve inventory turnover, and attract customers. However, excessive discounting can reduce profitability if the increase in sales volume does not compensate for lower margins.

The objective of this project was to determine:

- Whether discounts significantly influence sales
- How customer and product characteristics impact purchasing behaviour
- Which products generate stronger sales performance
- How data-driven pricing strategies can improve commercial decision-making

# Tools & Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Statsmodels
- Econometrics / OLS Regression

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv('synthetic_beverage_sales_data.csv')
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8999910 entries, 0 to 8999909
Data columns (total 11 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Order_ID       object 
 1   Customer_ID    object 
 2   Customer_Type  object 
 3   Product        object 
 4   Category       object 
 5   Unit_Price     float64
 6   Quantity       int64  
 7   Discount       float64
 8   Total_Price    float64
 9   Region         object 
 10  Order_Date     object 
dtypes: float64(3), int64(1), object(7)
memory usage: 755.3+ MB
None
  Order_ID Customer_ID Customer_Type             Product     Category  \
0     ORD1     CUS1496           B2B          Vio Wasser        Water   
1     ORD1     CUS1496           B2B               Evian        Water   
2     ORD1     CUS1496           B2B              Sprite  Soft Drinks   
3     ORD1     CUS1496           B2B  Rauch Multivitamin       Juices   
4     ORD1     CUS1496           B2B        Gerolsteiner        Water   

   Unit_Price  

# Dataset Information
The dataset contains approximately **8.99 million transaction records** from beverage sales.

## Key Variables

| Variable | Description |
|---|---|
| Order_ID | Unique order identifier |
| Customer_ID | Unique customer identifier |
| Customer_Type | B2B or B2C customer segment |
| Product | Beverage product sold |
| Category | Product category |
| Unit_Price | Price per unit |
| Quantity | Quantity purchased |
| Discount | Discount applied to the order |
| Total_Price | Final transaction value |
| Region | Sales region |
| Order_Date | Transaction date |


# Data Preparation

Several preprocessing steps were performed before building the regression model.

## Data Cleaning & Transformation

- Converted transaction dates into datetime format
- Removed unnecessary identifiers and non-modelling variables
- Created a logarithmic transformation of sales (`Log_sales`) to stabilize variance and improve interpretability
- Converted categorical variables into dummy variables using one-hot encoding
- Added an intercept term for regression modelling

In [3]:
# convert order_date dtype into datetime 
df['date_order'] = pd.to_datetime(df['Order_Date'])
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8999910 entries, 0 to 8999909
Data columns (total 12 columns):
 #   Column         Dtype         
---  ------         -----         
 0   Order_ID       object        
 1   Customer_ID    object        
 2   Customer_Type  object        
 3   Product        object        
 4   Category       object        
 5   Unit_Price     float64       
 6   Quantity       int64         
 7   Discount       float64       
 8   Total_Price    float64       
 9   Region         object        
 10  Order_Date     object        
 11  date_order     datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 824.0+ MB
None


## Feature Engineering

The target variable was defined as:

```python
Log_sales = log(Total_Price)
```

Using the logarithm of sales allows the model to measure percentage-based relationships and reduces the impact of extreme transaction values.


In [4]:
#build the sales sensitivity variable 
df['Log_sales']= np.log(df['Total_Price'])

In [5]:
# Dropping of the unwanted Col 
df1= df.drop(columns=['Order_ID', 'Customer_ID', 'Unit_Price', 'Order_Date','Total_Price', 'Region', 'Category', 'date_order'])
print(df1.info())
print(df1.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8999910 entries, 0 to 8999909
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Customer_Type  object 
 1   Product        object 
 2   Quantity       int64  
 3   Discount       float64
 4   Log_sales      float64
dtypes: float64(2), int64(1), object(2)
memory usage: 343.3+ MB
None
  Customer_Type             Product  Quantity  Discount  Log_sales
0           B2B          Vio Wasser        53      0.10   4.371724
1           B2B               Evian        90      0.10   4.839135
2           B2B              Sprite        73      0.05   4.396176
3           B2B  Rauch Multivitamin        59      0.10   5.141547
4           B2B        Gerolsteiner        35      0.10   3.310543


In [6]:
# conversion of Categorical variables in dummy vars 
df1= pd.get_dummies(df1, drop_first=True, dtype=int)
print(df1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8999910 entries, 0 to 8999909
Data columns (total 50 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Quantity                     int64  
 1   Discount                     float64
 2   Log_sales                    float64
 3   Customer_Type_B2C            int64  
 4   Product_Augustiner           int64  
 5   Product_Bacardi              int64  
 6   Product_Beck's               int64  
 7   Product_Chardonnay           int64  
 8   Product_Club Mate            int64  
 9   Product_Coca-Cola            int64  
 10  Product_Cranberry Juice      int64  
 11  Product_Erdinger Weißbier    int64  
 12  Product_Evian                int64  
 13  Product_Fanta                int64  
 14  Product_Fritz-Kola           int64  
 15  Product_Gerolsteiner         int64  
 16  Product_Granini Apple        int64  
 17  Product_Havana Club          int64  
 18  Product_Hohes C Orange       int64  
 19  

# Econometric Model

An Ordinary Least Squares (OLS) regression model was developed to estimate the impact of discounts and product characteristics on sales.

## Model Structure

The model estimated the following relationship:

\[
Log(Sales) = \beta_0 + \beta_1(Discount) + \beta_2(Quantity) + \beta_3(Customer\ Type) + \beta_n(Product\ Dummies) + \epsilon
\]

Where:

- `Log(Sales)` represents the logarithm of total transaction value
- `Discount` measures promotional reductions
- `Quantity` captures purchase volume
- `Customer Type` distinguishes B2B and B2C customers
- Product dummy variables capture product-level demand differences


In [7]:
import statsmodels.api as sm

In [ ]:
# Define independent variables (X)
X = df1.drop(columns=['Log_sales', 'Product_Augustiner', 'Product_Club Mate', 'Product_Cranberry Juice',
                     'Product_Gerolsteiner', 'Product_Mango Juice', 'Product_San Pellegrino'])

# Define target variable (y)
y = df1['Log_sales']

# Add constant (intercept)
X = sm.add_constant(X)

# Build OLS model
model = sm.OLS(y, X)

# Fit model
results = model.fit()
print(results.summary())

# Model Performance

| Metric | Value |
|---|---|
| Observations | 8,999,910 |
| R-squared | 0.848 |
| Adjusted R-squared | 0.848 |
| F-statistic | 1.169e+06 |

## Interpretation

The model explains approximately **84.8% of the variation in sales**, indicating very strong explanatory power.

The extremely high F-statistic suggests that the independent variables collectively provide strong predictive value for explaining sales behaviour.

---

# Key Insights

## 1. Discounts Significantly Impact Sales Revenue

The regression analysis shows that discounting has a substantial negative relationship with transaction revenue. Orders receiving larger discounts tend to generate significantly lower sales value after controlling for quantity purchased, customer type, and product characteristics. 

This suggests that aggressive discounting strategies may reduce profitability and should be carefully optimized to balance customer acquisition with revenue preservation.

---

## 2. Higher Purchase Volumes Drive Revenue Growth

The coefficient for `Quantity` indicates that each additional unit purchased is associated with an estimated **4.15% increase in sales revenue**, holding other variables constant. :contentReference[oaicite:1]{index=1}

This highlights the importance of bulk purchasing behaviour and larger order sizes in driving overall revenue performance.

---

## 3. B2B Customers Generate Higher Revenue Than B2C Customers

The model shows that B2C customers generate approximately **26.71% lower sales revenue** compared to B2B customers.

This indicates that commercial customers contribute more significantly to transaction value, likely due to higher purchasing volumes and repeat ordering behaviour.

---

## 4. Premium Alcohol Products Contribute Disproportionately to Revenue

Premium alcoholic beverages such as:

- Veuve Clicquot
- Moët & Chandon
- Johnnie Walker
- Jack Daniels
- Tanqueray
- Bacardi
- Havana Club

are associated with substantially higher transaction values relative to the baseline product category. 

These findings reflect the strong revenue contribution of premium spirits and champagne products within the beverage portfolio.

---

## 5. Lower-Priced Soft Drinks and Water Products Generate Smaller Transaction Values

Products such as:

- Coca-Cola
- Sprite
- Mountain Dew
- Volvic
- Vittel
- Selters

display significantly lower revenue contribution relative to premium beverage categories.

This reflects the lower unit pricing and lower transaction value typically associated with bottled water and standard soft drink products.

---

## 6. Energy Drinks and Juice Categories Show Strong Commercial Performance

The analysis indicates that energy drinks and juice products contribute positively to transaction revenue.

Products such as:

- Monster
- Red Bull
- Rockstar
- Passion Fruit Juice
- Tomato Juice
- Rauch Multivitamin

demonstrate stronger sales performance compared to several traditional soft drink categories.

This suggests growing commercial strength within functional beverage and energy drink segments.

---

## 7. The Econometric Model Demonstrates Strong Predictive Power

The OLS regression achieved an **R-squared of 0.848**, meaning the model explains approximately **84.8% of the variation in sales revenue**.

This demonstrates that pricing, quantity purchased, customer segmentation, and product selection collectively provide strong explanatory power for understanding sales behaviour and transaction performance.

---

# Business Implication 

## Pricing Strategy Optimisation

The results demonstrate that discounting can be used strategically to stimulate demand while monitoring profitability.

Retailers can use this type of analysis to identify:

- Optimal discount levels
- Products with high promotional responsiveness
- Products where discounts have limited impact
- Revenue-maximizing pricing strategies

---

## Customer Segmentation

The model highlights important behavioural differences between B2B and B2C customers.

This enables businesses to:

- Develop targeted pricing strategies
- Create customized promotions
- Improve loyalty campaigns
- Prioritize high-value customer segments

---

## Product Portfolio Management

The significant variation across products suggests opportunities for:

- Product prioritisation
- Dynamic pricing
- Inventory optimisation
- Promotion allocation

Products with stronger demand elasticity can receive more aggressive promotional campaigns.

---

## Revenue Forecasting Support

The regression framework can also support:

- Sales forecasting
- Promotional planning
- Revenue scenario analysis
- Demand sensitivity modelling

This creates a stronger analytical foundation for commercial decision-making.

---



# Conclusion

This project successfully applied econometric modelling techniques to evaluate the impact of discounts on sales performance using nearly 9 million beverage transactions.

The OLS regression model achieved strong explanatory performance with an R-squared of 0.848, demonstrating that pricing, customer segmentation, and product characteristics play a critical role in explaining sales outcomes.

The analysis highlights how data analytics and econometric modelling can support more effective pricing strategies, customer targeting, and revenue optimization.

Beyond measuring discount effectiveness, the project demonstrates the value of combining large-scale transactional data with statistical modelling to generate actionable business insights for retail and commercial decision-making.

---

# Skills Demonstrated

- Econometric Modelling
- OLS Regression
- Data Cleaning & Preprocessing
- Feature Engineering
- Dummy Variable Encoding
- Statistical Analysis
- Retail Sales Analytics
- Pricing Analytics
- Customer Behaviour Analysis
- Python for Data Analysis
- Business Insight Generation